### with the final cleaned dataset I am going to create a daily demand dataset which is the one that is going to be used for the model.

In [1]:
# ============================================================
# Daily Demand Cell 1
# Imports and file paths
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


# Use the final corrected file created in the earlier cells
TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)


# Main product-level daily forecasting dataset
DAILY_DEMAND_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_product_daily_demand_forecasting.csv"
)


# Product history and coverage summary
PRODUCT_SUMMARY_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_product_daily_demand_product_summary.csv"
)


# Dates treated as confirmed restaurant operating dates
OPERATING_DATES_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_operating_dates_used.csv"
)


# Audit showing the observed daily aggregation before zeros
DAILY_AGGREGATION_AUDIT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_product_daily_sales_aggregation_audit.csv"
)


DATA_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


if not TRANSACTION_INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Final corrected transaction file not found:\n"
        f"{TRANSACTION_INPUT_FILE}\n\n"
        "Run the previous saving cells first."
    )


print("Input file:")
print(TRANSACTION_INPUT_FILE)

print()
print("Main daily forecasting output:")
print(DAILY_DEMAND_OUTPUT_FILE)

Input file:
eden_datasets/UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv

Main daily forecasting output:
eden_datasets/UL_EDEN_product_daily_demand_forecasting.csv


In [2]:
# ============================================================
# Daily Demand Cell 2
# Load and validate the corrected transaction dataset
# ============================================================

transactions = pd.read_csv(
    TRANSACTION_INPUT_FILE
)


required_columns = [
    "TransDate",
    "TransValue",
    "PLUName",
    "GroupCode",
    "GroupName",
    "PLUCode",
    "Date",
    "Hour",
    "DayOfWeek",
    "Month",
    "WeekOfYear",
    "TransactionID",
    "UnitSold",
]


missing_columns = [
    column
    for column in required_columns
    if column not in transactions.columns
]


if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        f"{missing_columns}"
    )


# ------------------------------------------------------------
# Convert columns to appropriate data types
# ------------------------------------------------------------

transactions["TransDate"] = pd.to_datetime(
    transactions["TransDate"],
    errors="raise"
)


transactions["Date"] = (
    pd.to_datetime(
        transactions["Date"],
        errors="raise"
    )
    .dt.normalize()
)


transactions["TransValue"] = pd.to_numeric(
    transactions["TransValue"],
    errors="raise"
)


transactions["PLUCode"] = (
    pd.to_numeric(
        transactions["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


transactions["GroupCode"] = (
    pd.to_numeric(
        transactions["GroupCode"],
        errors="raise"
    )
    .astype("int64")
)


transactions["UnitSold"] = (
    pd.to_numeric(
        transactions["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


transactions["TransactionID"] = (
    transactions["TransactionID"]
    .astype("string")
    .str.strip()
)


# Clean product and group text again as a safety check
transactions["PLUName"] = (
    transactions["PLUName"]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)


transactions["GroupName"] = (
    transactions["GroupName"]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)


# ------------------------------------------------------------
# Validate the exact final dataset
# ------------------------------------------------------------

assert len(transactions) == 138_983, (
    "Expected 138,983 transaction rows."
)


assert int(
    transactions["UnitSold"].sum()
) == 141_480, (
    "Expected corrected UnitSold total of 141,480."
)


assert int(
    transactions["UnitSold"].gt(1).sum()
) == 61, (
    "Expected exactly 61 corrected multi-unit rows."
)


assert transactions["UnitSold"].ge(1).all(), (
    "UnitSold contains a value below 1."
)


assert transactions["UnitSold"].notna().all(), (
    "UnitSold contains missing values."
)


# Date must agree with the date part of TransDate
date_matches_transaction_date = (
    transactions["Date"]
    == transactions["TransDate"].dt.normalize()
)


assert date_matches_transaction_date.all(), (
    "At least one Date value does not match TransDate."
)


print("Final transaction dataset loaded and validated.")
print()
print("Rows:", f"{len(transactions):,}")
print(
    "Corrected units:",
    f"{int(transactions['UnitSold'].sum()):,}"
)
print(
    "Multi-unit transaction rows:",
    f"{int(transactions['UnitSold'].gt(1).sum()):,}"
)
print(
    "Date range:",
    transactions["Date"].min().date(),
    "to",
    transactions["Date"].max().date()
)

display(transactions.head())

Final transaction dataset loaded and validated.

Rows: 138,983
Corrected units: 141,480
Multi-unit transaction rows: 61
Date range: 2025-04-01 to 2026-03-30


,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode,Date,Hour,DayOfWeek,Month,WeekOfYear,TransactionID,UnitSold
0,2025-07-16 14:46:00,1026.0,VEGT MAINS 3,7,DINNER,4241483,2025-07-16,14,Wednesday,7,29,194814_2025-07-16_14-46-00,114
1,2025-07-17 14:14:00,549.0,MAINS 3,7,DINNER,4241480,2025-07-17,14,Thursday,7,29,194832_2025-07-17_14-14-00,61
2,2025-07-09 14:27:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-09,14,Wednesday,7,28,194782_2025-07-09_14-27-00,57
3,2025-07-18 14:35:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-18,14,Friday,7,29,194837_2025-07-18_14-35-00,57
4,2025-07-10 14:51:00,468.0,MAINS 3,7,DINNER,4241480,2025-07-10,14,Thursday,7,28,194801_2025-07-10_14-51-00,52


In [3]:
# ============================================================
# Daily Demand Cell 3
# Identify operating dates and exclude OPEN UL
# ============================================================

is_open_ul = (
    transactions["PLUName"]
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


# Use all recorded POS dates to determine restaurant operation
operating_dates = pd.DataFrame({
    "Date": sorted(
        transactions["Date"]
        .dropna()
        .unique()
    )
})


operating_dates["OperatingDayIndex"] = np.arange(
    1,
    len(operating_dates) + 1
)


# This records calendar gaps caused by weekends, closures,
# holidays or missing operating dates.
operating_dates[
    "DaysSincePreviousOperatingDay"
] = (
    operating_dates["Date"]
    .diff()
    .dt.days
    .fillna(0)
    .astype("int64")
)


# Exclude OPEN UL only from the forecastable products
forecast_transactions = transactions.loc[
    ~is_open_ul
].copy()


# ------------------------------------------------------------
# Validate expected operating data
# ------------------------------------------------------------

assert int(is_open_ul.sum()) == 24_736, (
    "Expected 24,736 OPEN UL rows."
)


assert len(operating_dates) == 245, (
    "Expected 245 observed operating dates."
)


assert forecast_transactions[
    "PLUCode"
].nunique() == 235, (
    "Expected 235 forecastable PLU codes."
)


assert int(
    forecast_transactions["UnitSold"].sum()
) == 116_744, (
    "Expected 116,744 units after excluding OPEN UL."
)


print("Operating dates and forecastable transactions prepared.")
print()
print(
    "Observed operating dates:",
    f"{len(operating_dates):,}"
)
print(
    "Forecastable products:",
    f"{forecast_transactions['PLUCode'].nunique():,}"
)
print(
    "OPEN UL rows excluded:",
    f"{int(is_open_ul.sum()):,}"
)
print(
    "Forecastable units:",
    f"{int(forecast_transactions['UnitSold'].sum()):,}"
)


operating_weekday_summary = (
    operating_dates["Date"]
    .dt.day_name()
    .value_counts()
    .reindex(
        [
            "Monday",
            "Tuesday",
            "Wednesday",
            "Thursday",
            "Friday",
            "Saturday",
            "Sunday",
        ],
        fill_value=0
    )
    .rename_axis("DayOfWeek")
    .reset_index(name="OperatingDates")
)


display(operating_weekday_summary)

Operating dates and forecastable transactions prepared.

Observed operating dates: 245
Forecastable products: 235
OPEN UL rows excluded: 24,736
Forecastable units: 116,744


,DayOfWeek,OperatingDates
0,Monday,45
1,Tuesday,50
2,Wednesday,50
3,Thursday,50
4,Friday,49
5,Saturday,1
6,Sunday,0


In [4]:
# ============================================================
# Daily Demand Cell 4
# Validate product metadata
# ============================================================

product_metadata_check = (
    forecast_transactions
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        ProductNameCount=(
            "PLUName",
            "nunique"
        ),
        GroupCodeCount=(
            "GroupCode",
            "nunique"
        ),
        GroupNameCount=(
            "GroupName",
            "nunique"
        ),
    )
)


inconsistent_products = product_metadata_check.loc[
    (
        product_metadata_check["ProductNameCount"].ne(1)
        | product_metadata_check["GroupCodeCount"].ne(1)
        | product_metadata_check["GroupNameCount"].ne(1)
    )
]


if len(inconsistent_products) > 0:
    display(inconsistent_products)

    raise ValueError(
        "Some PLU codes have inconsistent names or groups."
    )


# Create one metadata record per PLUCode
product_metadata = (
    forecast_transactions
    .sort_values(
        [
            "PLUCode",
            "Date",
            "TransDate",
        ]
    )
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        PLUName=(
            "PLUName",
            "first"
        ),
        GroupCode=(
            "GroupCode",
            "first"
        ),
        GroupName=(
            "GroupName",
            "first"
        ),
        FirstObservedSaleDate=(
            "Date",
            "min"
        ),
        LastObservedSaleDate=(
            "Date",
            "max"
        ),
    )
)


assert len(product_metadata) == 235, (
    "Product metadata should contain 235 PLU codes."
)


print("Product metadata validation passed.")
print(
    "Products:",
    len(product_metadata)
)

display(product_metadata.head())

Product metadata validation passed.
Products: 235


,PLUCode,PLUName,GroupCode,GroupName,FirstObservedSaleDate,LastObservedSaleDate
0,1031,TRAY BAKE,10,CAKES/PASTRIES,2025-10-14,2026-03-30
1,1313,MINCE PIE,10,CAKES/PASTRIES,2025-11-26,2026-03-20
2,1724,MUFFIN,10,CAKES/PASTRIES,2025-10-14,2026-03-30
3,2516,PORRIDGE/TOPPING,4,BREAKFAST,2025-11-25,2026-03-30
4,9102,SAUSAGE,4,BREAKFAST,2025-10-14,2026-03-30


In [5]:
# ============================================================
# Daily Demand Cell 5
# Aggregate observed demand by Date and PLUCode
# ============================================================

# These fields are for the aggregation audit only.
forecast_transactions[
    "_IsMultiUnitTransactionLine"
] = (
    forecast_transactions["UnitSold"]
    .gt(1)
    .astype("int64")
)


forecast_transactions[
    "_UnitsFromMultiUnitLines"
] = np.where(
    forecast_transactions["UnitSold"].gt(1),
    forecast_transactions["UnitSold"],
    0
)


daily_observed_sales = (
    forecast_transactions
    .groupby(
        [
            "Date",
            "PLUCode",
        ],
        as_index=False
    )
    .agg(
        DailyDemand=(
            "UnitSold",
            "sum"
        ),
        TransactionLineCount=(
            "TransactionID",
            "size"
        ),
        UniqueReceiptCount=(
            "TransactionID",
            "nunique"
        ),
        DailyTransValue=(
            "TransValue",
            "sum"
        ),
        MultiUnitTransactionRows=(
            "_IsMultiUnitTransactionLine",
            "sum"
        ),
        UnitsFromMultiUnitRows=(
            "_UnitsFromMultiUnitLines",
            "sum"
        ),
    )
)


daily_observed_sales["DailyDemand"] = (
    daily_observed_sales["DailyDemand"]
    .astype("int64")
)


daily_observed_sales["DailyTransValue"] = (
    daily_observed_sales["DailyTransValue"]
    .round(2)
)


# Add readable product information
daily_observed_sales = daily_observed_sales.merge(
    product_metadata[
        [
            "PLUCode",
            "PLUName",
            "GroupCode",
            "GroupName",
        ]
    ],
    on="PLUCode",
    how="left",
    validate="many_to_one"
)


daily_observed_sales = daily_observed_sales[
    [
        "Date",
        "PLUCode",
        "PLUName",
        "GroupCode",
        "GroupName",
        "DailyDemand",
        "TransactionLineCount",
        "UniqueReceiptCount",
        "DailyTransValue",
        "MultiUnitTransactionRows",
        "UnitsFromMultiUnitRows",
    ]
].sort_values(
    [
        "Date",
        "PLUCode",
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# Validate observed daily aggregation
# ------------------------------------------------------------

assert len(daily_observed_sales) == 15_139, (
    "Expected 15,139 observed product-date sales rows."
)


assert int(
    daily_observed_sales["DailyDemand"].sum()
) == 116_744, (
    "Observed daily demand does not equal the "
    "forecastable transaction-unit total."
)


assert not daily_observed_sales.duplicated(
    [
        "Date",
        "PLUCode",
    ]
).any(), (
    "Observed daily aggregation contains duplicate keys."
)


print("Observed transaction rows aggregated successfully.")
print()
print(
    "Observed product-date rows:",
    f"{len(daily_observed_sales):,}"
)
print(
    "Observed daily-demand total:",
    f"{int(daily_observed_sales['DailyDemand'].sum()):,}"
)


display(
    daily_observed_sales
    .sort_values(
        "DailyDemand",
        ascending=False
    )
    .head(20)
)

Observed transaction rows aggregated successfully.

Observed product-date rows: 15,139
Observed daily-demand total: 116,744


,Date,PLUCode,PLUName,GroupCode,GroupName,DailyDemand,TransactionLineCount,UniqueReceiptCount,DailyTransValue,MultiUnitTransactionRows,UnitsFromMultiUnitRows
7742,2025-09-24,4241428,HAM & CHEESE SANDWICH CT,5,SANDWICHES,309,9,5,1545.0,5,305
7840,2025-09-26,4241428,HAM & CHEESE SANDWICH CT,5,SANDWICHES,306,6,2,1530.0,5,305
8063,2025-10-02,4241476,KIMBOX MAINS 2,7,DINNER,208,208,203,1456.0,0,0
9874,2025-11-18,42533,€7 KIMBOCK,7,DINNER,195,195,194,1365.0,0,0
8369,2025-10-13,4241476,KIMBOX MAINS 2,7,DINNER,185,185,179,1295.0,0,0
10249,2025-11-26,42533,€7 KIMBOCK,7,DINNER,184,184,183,1288.0,0,0
8198,2025-10-07,4241476,KIMBOX MAINS 2,7,DINNER,181,181,180,1267.0,0,0
8242,2025-10-08,4241476,KIMBOX MAINS 2,7,DINNER,180,180,177,1260.0,0,0
9016,2025-10-29,42533,€7 KIMBOCK,7,DINNER,180,180,175,1260.0,0,0
10314,2025-11-27,42533,€7 KIMBOCK,7,DINNER,179,179,173,1253.0,0,0


In [6]:
# ============================================================
# Daily Demand Cell 6
# Create active product-date panel and add zero demand
# ============================================================

# Create every possible product and operating-date combination
product_date_panel = (
    product_metadata
    .assign(_JoinKey=1)
    .merge(
        operating_dates.assign(
            _JoinKey=1
        ),
        on="_JoinKey",
        how="inner"
    )
    .drop(
        columns="_JoinKey"
    )
)


# Keep only operating dates within each product's observed
# first-to-last sale window.
product_date_panel = product_date_panel.loc[
    (
        product_date_panel["Date"]
        >= product_date_panel["FirstObservedSaleDate"]
    )
    &
    (
        product_date_panel["Date"]
        <= product_date_panel["LastObservedSaleDate"]
    )
].copy()


# Merge the observed daily demand
product_date_panel = product_date_panel.merge(
    daily_observed_sales[
        [
            "Date",
            "PLUCode",
            "DailyDemand",
        ]
    ],
    on=[
        "Date",
        "PLUCode",
    ],
    how="left",
    validate="one_to_one"
)


# A missing value now means:
# restaurant operated, product was within its observed
# sales window, but no sale was recorded for that product.
product_date_panel["DailyDemand"] = (
    product_date_panel["DailyDemand"]
    .fillna(0)
    .astype("int64")
)


# Sort temporarily by product and date for product indexes
product_date_panel = (
    product_date_panel
    .sort_values(
        [
            "PLUCode",
            "Date",
        ]
    )
    .reset_index(drop=True)
)


product_date_panel[
    "ProductOperatingDayIndex"
] = (
    product_date_panel
    .groupby("PLUCode")
    .cumcount()
    + 1
)


product_date_panel[
    "ProductAgeCalendarDays"
] = (
    product_date_panel["Date"]
    - product_date_panel["FirstObservedSaleDate"]
).dt.days.astype("int64")


# ------------------------------------------------------------
# Validate the zero-filled panel
# ------------------------------------------------------------

assert len(product_date_panel) == 25_407, (
    "Expected 25,407 active product-date rows."
)


assert int(
    product_date_panel["DailyDemand"].sum()
) == 116_744, (
    "Zero filling changed the total demand."
)


assert int(
    product_date_panel["DailyDemand"].eq(0).sum()
) == 10_268, (
    "Expected 10,268 zero-demand product-date rows."
)


assert not product_date_panel.duplicated(
    [
        "Date",
        "PLUCode",
    ]
).any(), (
    "Product-date panel contains duplicate keys."
)


print("Product-date panel created successfully.")
print()
print(
    "Product-date rows:",
    f"{len(product_date_panel):,}"
)
print(
    "Rows with observed demand:",
    f"{int(product_date_panel['DailyDemand'].gt(0).sum()):,}"
)
print(
    "Zero-demand rows added:",
    f"{int(product_date_panel['DailyDemand'].eq(0).sum()):,}"
)

Product-date panel created successfully.

Product-date rows: 25,407
Rows with observed demand: 15,139
Zero-demand rows added: 10,268


In [7]:
# ============================================================
# Daily Demand Cell 7
# Add calendar and sequence features
# ============================================================

iso_calendar = (
    product_date_panel["Date"]
    .dt.isocalendar()
)


product_date_panel["Year"] = (
    product_date_panel["Date"]
    .dt.year
    .astype("int64")
)


product_date_panel["Month"] = (
    product_date_panel["Date"]
    .dt.month
    .astype("int64")
)


product_date_panel["Quarter"] = (
    product_date_panel["Date"]
    .dt.quarter
    .astype("int64")
)


product_date_panel["DayOfMonth"] = (
    product_date_panel["Date"]
    .dt.day
    .astype("int64")
)


# Monday = 0 and Sunday = 6
product_date_panel["DayOfWeekNumber"] = (
    product_date_panel["Date"]
    .dt.dayofweek
    .astype("int64")
)


product_date_panel["DayOfWeek"] = (
    product_date_panel["Date"]
    .dt.day_name()
)


product_date_panel["ISOYear"] = (
    iso_calendar["year"]
    .astype("int64")
)


product_date_panel["WeekOfYear"] = (
    iso_calendar["week"]
    .astype("int64")
)


product_date_panel["IsWeekend"] = (
    product_date_panel["DayOfWeekNumber"]
    .isin(
        [
            5,
            6,
        ]
    )
    .astype("int64")
)


product_date_panel["IsMonthStart"] = (
    product_date_panel["Date"]
    .dt.is_month_start
    .astype("int64")
)


product_date_panel["IsMonthEnd"] = (
    product_date_panel["Date"]
    .dt.is_month_end
    .astype("int64")
)


print("Calendar features added.")

Calendar features added.


In [8]:
# ============================================================
# Daily Demand Cell 8
# Create final model-ready daily demand dataset
# ============================================================

daily_demand_df = product_date_panel[
    [
        "Date",
        "OperatingDayIndex",
        "DaysSincePreviousOperatingDay",
        "PLUCode",
        "PLUName",
        "GroupCode",
        "GroupName",
        "DailyDemand",
        "ProductOperatingDayIndex",
        "ProductAgeCalendarDays",
        "Year",
        "Month",
        "Quarter",
        "DayOfMonth",
        "DayOfWeekNumber",
        "DayOfWeek",
        "ISOYear",
        "WeekOfYear",
        "IsWeekend",
        "IsMonthStart",
        "IsMonthEnd",
    ]
].copy()


daily_demand_df = (
    daily_demand_df
    .sort_values(
        [
            "Date",
            "PLUCode",
        ]
    )
    .reset_index(drop=True)
)


print("Final daily demand dataset created.")
print()
print("Rows:", f"{len(daily_demand_df):,}")
print("Columns:", len(daily_demand_df.columns))

display(daily_demand_df.head(20))

Final daily demand dataset created.

Rows: 25,407
Columns: 21


,Date,OperatingDayIndex,DaysSincePreviousOperatingDay,PLUCode,PLUName,GroupCode,GroupName,DailyDemand,ProductOperatingDayIndex,ProductAgeCalendarDays,...,Month,Quarter,DayOfMonth,DayOfWeekNumber,DayOfWeek,ISOYear,WeekOfYear,IsWeekend,IsMonthStart,IsMonthEnd
0,2025-04-01,1,0,12304,KOMBUCHA,2,COLD BEVS,1,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
1,2025-04-01,1,0,31331,FILTER COFFEE LARGE,1,HOT BEVS,1,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
2,2025-04-01,1,0,125038,FILTER COFFEE SM,1,HOT BEVS,5,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
3,2025-04-01,1,0,3219121,CARTON OF WATER,2,COLD BEVS,5,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
4,2025-04-01,1,0,3570509,FULFIL BARS,15,HOSPITALITY,2,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
5,2025-04-01,1,0,4241401,TAYTO CHEESE,33,SNACKS,3,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
6,2025-04-01,1,0,4241402,WALKERS BAKED,33,SNACKS,1,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
7,2025-04-01,1,0,4241404,ODONNELL /POPCORN,33,SNACKS,8,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
8,2025-04-01,1,0,4241407,PURPLE SNACK,33,SNACKS,2,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
9,2025-04-01,1,0,4241408,CHOCOLATE BARS,33,SNACKS,13,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0


In [9]:
# ============================================================
# Daily Demand Cell 9
# Final validation of daily forecasting dataset
# ============================================================

# ------------------------------------------------------------
# Structural checks
# ------------------------------------------------------------

assert len(daily_demand_df) == 25_407, (
    "Final daily dataset should contain 25,407 rows."
)


assert daily_demand_df[
    "PLUCode"
].nunique() == 235, (
    "Final dataset should contain 235 PLU codes."
)


assert daily_demand_df[
    "Date"
].nunique() == 245, (
    "Final dataset should contain 245 operating dates."
)


assert not daily_demand_df.duplicated(
    [
        "Date",
        "PLUCode",
    ]
).any(), (
    "Final dataset contains duplicate Date + PLUCode keys."
)


# ------------------------------------------------------------
# Target checks
# ------------------------------------------------------------

assert daily_demand_df[
    "DailyDemand"
].notna().all(), (
    "DailyDemand contains missing values."
)


assert daily_demand_df[
    "DailyDemand"
].ge(0).all(), (
    "DailyDemand contains a negative value."
)


assert np.allclose(
    daily_demand_df[
        "DailyDemand"
    ].to_numpy(dtype=float),
    np.round(
        daily_demand_df[
            "DailyDemand"
        ].to_numpy(dtype=float)
    )
), (
    "DailyDemand contains a non-whole-number value."
)


assert int(
    daily_demand_df["DailyDemand"].sum()
) == 116_744, (
    "Final DailyDemand total should be 116,744."
)


positive_demand_rows = int(
    daily_demand_df[
        "DailyDemand"
    ].gt(0).sum()
)


zero_demand_rows = int(
    daily_demand_df[
        "DailyDemand"
    ].eq(0).sum()
)


assert positive_demand_rows == 15_139, (
    "Expected 15,139 rows with positive demand."
)


assert zero_demand_rows == 10_268, (
    "Expected 10,268 zero-demand rows."
)


# ------------------------------------------------------------
# Verify that observed sales were preserved exactly
# ------------------------------------------------------------

observed_demand_series = (
    daily_observed_sales
    .set_index(
        [
            "Date",
            "PLUCode",
        ]
    )["DailyDemand"]
    .sort_index()
)


panel_positive_demand_series = (
    daily_demand_df.loc[
        daily_demand_df["DailyDemand"].gt(0)
    ]
    .set_index(
        [
            "Date",
            "PLUCode",
        ]
    )["DailyDemand"]
    .sort_index()
)


assert observed_demand_series.equals(
    panel_positive_demand_series
), (
    "Observed product-date demand changed during "
    "zero-demand panel creation."
)


# ------------------------------------------------------------
# Ensure OPEN UL is absent
# ------------------------------------------------------------

assert not (
    daily_demand_df["PLUName"]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
).any(), (
    "OPEN UL is present in the daily demand dataset."
)


print("All daily forecasting dataset checks passed.")
print()
print(
    "Product-day rows:",
    f"{len(daily_demand_df):,}"
)
print(
    "Products:",
    f"{daily_demand_df['PLUCode'].nunique():,}"
)
print(
    "Operating dates:",
    f"{daily_demand_df['Date'].nunique():,}"
)
print(
    "Positive-demand rows:",
    f"{positive_demand_rows:,}"
)
print(
    "Zero-demand rows:",
    f"{zero_demand_rows:,}"
)
print(
    "Total forecastable demand:",
    f"{int(daily_demand_df['DailyDemand'].sum()):,}"
)

All daily forecasting dataset checks passed.

Product-day rows: 25,407
Products: 235
Operating dates: 245
Positive-demand rows: 15,139
Zero-demand rows: 10,268
Total forecastable demand: 116,744


In [10]:
# ============================================================
# Daily Demand Cell 10
# Create product forecasting coverage summary
# ============================================================

product_summary = (
    daily_demand_df
    .groupby(
        [
            "PLUCode",
            "PLUName",
            "GroupCode",
            "GroupName",
        ],
        as_index=False
    )
    .agg(
        FirstObservedSaleDate=(
            "Date",
            "min"
        ),
        LastObservedSaleDate=(
            "Date",
            "max"
        ),
        ActiveOperatingDays=(
            "Date",
            "nunique"
        ),
        DaysWithSales=(
            "DailyDemand",
            lambda values: int(
                values.gt(0).sum()
            )
        ),
        ZeroDemandDays=(
            "DailyDemand",
            lambda values: int(
                values.eq(0).sum()
            )
        ),
        TotalDemand=(
            "DailyDemand",
            "sum"
        ),
        MeanDemandPerActiveOperatingDay=(
            "DailyDemand",
            "mean"
        ),
        MedianDemandPerActiveOperatingDay=(
            "DailyDemand",
            "median"
        ),
        MaximumDailyDemand=(
            "DailyDemand",
            "max"
        ),
    )
)


mean_demand_when_sold = (
    daily_demand_df.loc[
        daily_demand_df["DailyDemand"].gt(0)
    ]
    .groupby("PLUCode")["DailyDemand"]
    .mean()
    .rename("MeanDemandWhenSold")
    .reset_index()
)


product_summary = product_summary.merge(
    mean_demand_when_sold,
    on="PLUCode",
    how="left",
    validate="one_to_one"
)


product_summary["CalendarSpanDays"] = (
    product_summary["LastObservedSaleDate"]
    - product_summary["FirstObservedSaleDate"]
).dt.days + 1


product_summary["ZeroDemandRate"] = (
    product_summary["ZeroDemandDays"]
    / product_summary["ActiveOperatingDays"]
).round(4)


product_summary["HistoryLengthCategory"] = pd.cut(
    product_summary["ActiveOperatingDays"],
    bins=[
        0,
        29,
        59,
        119,
        np.inf,
    ],
    labels=[
        "Very short (<30)",
        "Short (30-59)",
        "Medium (60-119)",
        "Long (120+)",
    ]
)


product_summary = (
    product_summary
    .sort_values(
        [
            "TotalDemand",
            "ActiveOperatingDays",
        ],
        ascending=[
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)


assert len(product_summary) == 235

assert int(
    product_summary["TotalDemand"].sum()
) == 116_744

assert int(
    product_summary["ActiveOperatingDays"].sum()
) == 25_407


print("Product coverage summary created.")
print()
print(
    "Products:",
    len(product_summary)
)


history_category_summary = (
    product_summary[
        "HistoryLengthCategory"
    ]
    .value_counts(dropna=False)
    .rename_axis("HistoryLengthCategory")
    .reset_index(name="Products")
)


display(history_category_summary)

display(product_summary.head(20))

Product coverage summary created.

Products: 235


,HistoryLengthCategory,Products
0,Medium (60-119),143
1,Long (120+),65
2,Very short (<30),18
3,Short (30-59),9


,PLUCode,PLUName,GroupCode,GroupName,FirstObservedSaleDate,LastObservedSaleDate,ActiveOperatingDays,DaysWithSales,ZeroDemandDays,TotalDemand,MeanDemandPerActiveOperatingDay,MedianDemandPerActiveOperatingDay,MaximumDailyDemand,MeanDemandWhenSold,CalendarSpanDays,ZeroDemandRate,HistoryLengthCategory
0,42533,€7 KIMBOCK,7,DINNER,2025-10-14,2026-03-30,110,101,9,10946,99.509091,111.0,195,108.376238,168,0.0818,Medium (60-119)
1,4241476,KIMBOX MAINS 2,7,DINNER,2025-04-01,2025-10-13,135,128,7,6623,49.059259,37.0,208,51.742188,196,0.0519,Long (120+)
2,200000003,SUGAR FREE CAN,2,COLD BEVS,2025-04-02,2026-03-30,244,212,32,3593,14.725410,5.0,77,16.948113,363,0.1311,Long (120+)
3,2000000019,FULL FAT CAN,2,COLD BEVS,2025-04-01,2026-03-30,245,245,0,3358,13.706122,11.0,94,13.706122,364,0.0000,Long (120+)
4,42529,€5.00 DINNER,7,DINNER,2025-10-14,2026-03-30,110,109,1,3312,30.109091,29.0,87,30.385321,168,0.0091,Medium (60-119)
5,42530,€7 DINNER,7,DINNER,2025-10-14,2026-03-30,110,102,8,3192,29.018182,28.5,103,31.294118,168,0.0727,Medium (60-119)
6,4241512,TEA,1,HOT BEVS,2025-10-14,2026-03-30,110,110,0,3067,27.881818,29.0,56,27.881818,168,0.0000,Medium (60-119)
7,44381,RN AMERICANO,1,HOT BEVS,2025-10-14,2026-03-30,110,110,0,2935,26.681818,27.0,51,26.681818,168,0.0000,Medium (60-119)
8,4241408,CHOCOLATE BARS,33,SNACKS,2025-04-01,2026-03-30,245,244,1,2902,11.844898,11.0,46,11.893443,364,0.0041,Long (120+)
9,3219121,CARTON OF WATER,2,COLD BEVS,2025-04-01,2026-03-30,245,244,1,2698,11.012245,9.0,62,11.057377,364,0.0041,Long (120+)


In [11]:
# ============================================================
# Daily Demand Cell 11
# Save daily demand and supporting audit files
# ============================================================

daily_demand_df.to_csv(
    DAILY_DEMAND_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


product_summary.to_csv(
    PRODUCT_SUMMARY_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


operating_dates.to_csv(
    OPERATING_DATES_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


daily_observed_sales.to_csv(
    DAILY_AGGREGATION_AUDIT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("Daily demand files saved successfully.")
print()

print("1. Main daily forecasting dataset:")
print(DAILY_DEMAND_OUTPUT_FILE)
print()

print("2. Product coverage summary:")
print(PRODUCT_SUMMARY_OUTPUT_FILE)
print()

print("3. Operating dates used:")
print(OPERATING_DATES_OUTPUT_FILE)
print()

print("4. Observed daily aggregation audit:")
print(DAILY_AGGREGATION_AUDIT_FILE)

Daily demand files saved successfully.

1. Main daily forecasting dataset:
eden_datasets/UL_EDEN_product_daily_demand_forecasting.csv

2. Product coverage summary:
eden_datasets/UL_EDEN_product_daily_demand_product_summary.csv

3. Operating dates used:
eden_datasets/UL_EDEN_operating_dates_used.csv

4. Observed daily aggregation audit:
eden_datasets/UL_EDEN_product_daily_sales_aggregation_audit.csv


In [12]:
# ============================================================
# Daily Demand Cell 12
# Confirm output files exist
# ============================================================

daily_output_files = {
    "Daily forecasting dataset":
        DAILY_DEMAND_OUTPUT_FILE,

    "Product coverage summary":
        PRODUCT_SUMMARY_OUTPUT_FILE,

    "Operating dates":
        OPERATING_DATES_OUTPUT_FILE,

    "Daily aggregation audit":
        DAILY_AGGREGATION_AUDIT_FILE,
}


missing_daily_files = [
    str(file_path)
    for file_path in daily_output_files.values()
    if not file_path.exists()
]


if missing_daily_files:
    raise FileNotFoundError(
        "The following files were not created:\n"
        + "\n".join(missing_daily_files)
    )


print("All expected daily-demand files exist.")
print()


for description, file_path in daily_output_files.items():

    file_size_mb = (
        file_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{description}: "
        f"{file_path} "
        f"({file_size_mb:.2f} MB)"
    )

All expected daily-demand files exist.

Daily forecasting dataset: eden_datasets/UL_EDEN_product_daily_demand_forecasting.csv (2.32 MB)
Product coverage summary: eden_datasets/UL_EDEN_product_daily_demand_product_summary.csv (0.03 MB)
Operating dates: eden_datasets/UL_EDEN_operating_dates_used.csv (0.00 MB)
Daily aggregation audit: eden_datasets/UL_EDEN_product_daily_sales_aggregation_audit.csv (0.87 MB)


In [13]:
# ============================================================
# Daily Demand Cell 13
# Reload and validate saved daily forecasting dataset
# ============================================================

saved_daily_demand = pd.read_csv(
    DAILY_DEMAND_OUTPUT_FILE,
    parse_dates=["Date"]
)


saved_daily_demand["PLUCode"] = (
    pd.to_numeric(
        saved_daily_demand["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


saved_daily_demand["DailyDemand"] = (
    pd.to_numeric(
        saved_daily_demand["DailyDemand"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


assert len(saved_daily_demand) == 25_407, (
    "Saved daily-demand row count is incorrect."
)


assert saved_daily_demand[
    "PLUCode"
].nunique() == 235, (
    "Saved product count is incorrect."
)


assert saved_daily_demand[
    "Date"
].nunique() == 245, (
    "Saved operating-date count is incorrect."
)


assert int(
    saved_daily_demand["DailyDemand"].sum()
) == 116_744, (
    "Saved DailyDemand total is incorrect."
)


assert int(
    saved_daily_demand["DailyDemand"].eq(0).sum()
) == 10_268, (
    "Saved zero-demand row count is incorrect."
)


assert not saved_daily_demand.duplicated(
    [
        "Date",
        "PLUCode",
    ]
).any(), (
    "Saved dataset contains duplicate product-date rows."
)


assert saved_daily_demand[
    "DailyDemand"
].ge(0).all(), (
    "Saved DailyDemand contains a negative value."
)


assert not (
    saved_daily_demand["PLUName"]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
).any(), (
    "OPEN UL is present in the saved dataset."
)


print("Saved daily forecasting dataset validation passed.")
print()
print("Rows:", f"{len(saved_daily_demand):,}")
print(
    "Products:",
    f"{saved_daily_demand['PLUCode'].nunique():,}"
)
print(
    "Operating dates:",
    f"{saved_daily_demand['Date'].nunique():,}"
)
print(
    "Total demand:",
    f"{int(saved_daily_demand['DailyDemand'].sum()):,}"
)
print(
    "Zero-demand rows:",
    f"{int(saved_daily_demand['DailyDemand'].eq(0).sum()):,}"
)

display(saved_daily_demand.head(20))

Saved daily forecasting dataset validation passed.

Rows: 25,407
Products: 235
Operating dates: 245
Total demand: 116,744
Zero-demand rows: 10,268


,Date,OperatingDayIndex,DaysSincePreviousOperatingDay,PLUCode,PLUName,GroupCode,GroupName,DailyDemand,ProductOperatingDayIndex,ProductAgeCalendarDays,...,Month,Quarter,DayOfMonth,DayOfWeekNumber,DayOfWeek,ISOYear,WeekOfYear,IsWeekend,IsMonthStart,IsMonthEnd
0,2025-04-01,1,0,12304,KOMBUCHA,2,COLD BEVS,1,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
1,2025-04-01,1,0,31331,FILTER COFFEE LARGE,1,HOT BEVS,1,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
2,2025-04-01,1,0,125038,FILTER COFFEE SM,1,HOT BEVS,5,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
3,2025-04-01,1,0,3219121,CARTON OF WATER,2,COLD BEVS,5,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
4,2025-04-01,1,0,3570509,FULFIL BARS,15,HOSPITALITY,2,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
5,2025-04-01,1,0,4241401,TAYTO CHEESE,33,SNACKS,3,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
6,2025-04-01,1,0,4241402,WALKERS BAKED,33,SNACKS,1,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
7,2025-04-01,1,0,4241404,ODONNELL /POPCORN,33,SNACKS,8,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
8,2025-04-01,1,0,4241407,PURPLE SNACK,33,SNACKS,2,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
9,2025-04-01,1,0,4241408,CHOCOLATE BARS,33,SNACKS,13,1,0,...,4,2,1,1,Tuesday,2025,14,0,1,0
